Dylan Ross

Select out subsets of the omics data based on targeted pairwise comparisons:
- Black patients: NPM1 mutant vs. WT
- White patients: NPM1 mutant vs. WT
- Black patients: NRAS mutant vs. WT
- White patients: NRAS mutant vs. WT

## Setup

### Imports

In [1]:
import os
from typing import List
import pickle

import polars as pl
from scipy import stats
from plotly import graph_objects as pgo

### Constants

In [2]:
# Jupyter server should be running from src/python
CACHE_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "dylan",
    "_cache"
)

FIGURE_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "dylan",
    "_figures"
)

# cached metadata table 
META_CACHE = os.path.join(CACHE_DIR, "meta.arrow")
OMICS_CACHE = os.path.join(CACHE_DIR, "combined.arrow")

# missingness threshold (as a proportion) for dropping omics data rows
# rows with missingness above this threshold are dropped
MISSING_THRESHOLD = 0.3

### Load metadata and omics data from cache

In [3]:
meta = pl.read_ipc(META_CACHE)
meta

Sample,Age,Sex,Race,Study,FLT3_ITD,IDH1+IDH2,NPM1,NRAS
str,f64,str,str,str,bool,bool,bool,bool
"""11-00261""",74.0,"""Male""","""White""","""BeatAML""",true,false,false,false
"""11-00503""",54.0,"""Female""","""White""","""BeatAML""",true,false,true,false
"""11-00475""",65.0,"""Male""","""White""","""BeatAML""",true,false,true,false
"""12-00032""",70.0,"""Male""","""White""","""BeatAML""",true,false,false,false
"""11-00376""",49.0,"""Male""","""White""","""BeatAML""",true,false,true,false
…,…,…,…,…,…,…,…,…
"""16-01109-Bridge""",33.0,"""Male""","""Black""","""pilotStudy""",false,false,false,false
"""16-01191-Bridge""",65.0,"""Female""","""White""","""pilotStudy""",false,false,false,true
"""17-00025-Bridge""",39.0,"""Female""","""White""","""pilotStudy""",true,false,true,false


In [4]:
combined = pl.read_ipc(OMICS_CACHE)
combined

Block,Feature,11-00261,11-00503,11-00475,12-00032,11-00376,11-00378,11-00382,11-00388,11-00416,11-00465,11-00466,12-00123,12-00127,12-00145,12-00196,12-00294,12-00383,13-00033,13-00034,13-00075,13-00077,13-00123,13-00149,13-00157,13-00160,13-00186,13-00195,13-00226,13-00262,13-00331,13-00450,13-00468,14-00495,13-00558,13-00581,…,C-95-068,C-01-1665,C-09-1033,C-99-2136,C-05-1782,94-C-376,PS88-0050,C-01-2163,C-02-0350,C-04-0434,C-01-0171,PS88-0140,C-11-2295,C-99-1077,C-05-0004,C-05-3159,C-05-4372,PS89-0158,C-01-0930,C-98-0031,94-C-077,C-98-0033,C-98-0665,C-99-1700,C-02-1356,C-99-2065,14-00528-Bridge,16-00120-Bridge,16-00494-Bridge,16-00627-Bridge,16-00731-Bridge,16-01100-Bridge,16-01109-Bridge,16-01191-Bridge,17-00025-Bridge,17-00741-Bridge,17-00881-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Acetylomics""","""A2M-K1176k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,0.330127,-1.505467,-0.428963,null,-2.143029,null,-0.497628,-1.37027,-1.787643,null,-0.510944,null,2.104951,-0.382382,-0.304169,null,1.271643,null,0.313458,2.126205,-1.622013,null,0.880241,null,-2.93503,null,null,null,null,null,null,null,null,null,null,null
"""Acetylomics""","""A2M-K912k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,0.38503,0.695151,-0.219241,null,0.912355,null,null,0.890893,0.79203,0.001479,-1.447282,null,0.818989,-0.03042,null,null,0.277311,null,0.435211,0.545431,0.38928,null,0.454476,null,-0.020518,null,null,null,null,null,null,null,null,null,null,null
"""Acetylomics""","""ABCE1-K343k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,-1.260764,-0.196359,0.365935,null,-0.641335,null,-0.119836,0.099939,0.551293,null,-1.03373,null,0.41977,-0.715829,-0.281123,null,-0.325684,null,0.658935,0.268563,-1.661094,null,-1.71928,null,0.094878,0.317464,null,null,null,null,null,null,null,null,null,null,null
"""Acetylomics""","""ABCE1-K431k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,-1.354356,0.582544,0.370127,-0.031272,-0.086364,0.327699,-0.292101,-1.60355,-0.159053,-0.481542,-0.750778,-0.383407,0.373132,-0.205401,-0.688846,-0.799921,-0.366197,-0.039556,0.701408,-0.181248,-0.911242,-1.698226,1.016596,-0.274447,-0.361193,-0.49722,null,null,null,null,null,null,null,null,null,null,null
"""Acetylomics""","""ABHD10-K69k""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,0.512305,-0.641897,null,-0.788261,null,-1.523059,-0.880812,-0.147372,null,-0.379555,null,0.040963,1.265502,0.47155,null,-1.411947,null,-0.87768,-0.02174,-1.051302,null,0.591655,null,-0.609805,0.167879,-1.579952,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Transcriptomics""","""CEP43__2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.371185,null,null,0.0,null,null,0.087719,0.249785,0.155027,0.099639,null,0.155454,0.156892,0.038594,0.185828,0.141952,0.126895,0.206017,0.094288,0.241965,…,0.077266,0.146562,0.540351,null,null,0.038757,null,0.069963,0.088275,0.338344,0.088325,null,null,0.350199,0.1

In [5]:
# report on the number of features in each block
combined.group_by("Block").len().sort("len", descending=True)

Block,len
str,u32
"""Transcriptomics""",14372
"""Phosphoproteomics""",8362
"""Proteomics""",8059
"""Acetylomics""",3285
"""Lipidomics""",343
"""Metabolomics""",49


## Create Subsets

### Utility function for selecting subsets and applying transformations/normalizations
- filter columns to only include the selected samples for the subset (with two sample groups)
- filter rows separately for each sample group to remove rows with missingness fraction greater than a specified threshold
- impute remaining missing values across all subset samples with row median
- apply row-wise z-score normalization across subset samples 

In [6]:
def select_subset(
    combined: pl.DataFrame,
    wt_cols: List[str],
    mut_cols: List[str]
) -> pl.DataFrame :
    """
    Select a subset of the combined -omics data DataFrame that contains a specified
    set of samples belonging to WT/mutant groups. Filters rows based on a missingness
    threshold, imputes remaining missing values with row median, then applies a 
    row-wise z-score normalization to the full table. Returns the processed DataFrame
    with the selected subset.
    """
    return (
        combined
        .select(
            "Block",
            "Feature",
            *wt_cols,
            *mut_cols
        )
        .filter(
            # filter out rows where missingness within WT or mut sample groups
            # is greater than missingness threshold
            ((pl.sum_horizontal(pl.col(wt_cols).is_null()) / len(wt_cols)) <= MISSING_THRESHOLD) 
            & ((pl.sum_horizontal(pl.col(mut_cols).is_null()) / len(mut_cols)) <= MISSING_THRESHOLD)
        )
        .with_columns(
            # add a column with the row median
            pl.concat_list(wt_cols + mut_cols)
            .list.median()
            .alias("row_median")
        )
        .select(
            "Block",
            "Feature",
            (
                # pack the raw values into a list column
                pl.concat_list(
                    pl.col(wt_cols + mut_cols)
                    # fill null values with row median
                    .fill_null(pl.col("row_median"))
                )
                # compute row-wise z-scores
                .map_elements(
                    stats.zscore, 
                    return_dtype=pl.List(pl.Float64)
                )
                # convert list to struct 
                .list.to_struct(
                    fields=wt_cols + mut_cols
                )
                .alias("packed")
            )
        )
        # unpack struct with z-scores back to sample columns
        .unnest("packed")
    )

### Utility function to plot feature counts from the -omics data blocks before/after subsetting

In [19]:
def plot_filtered_feature_counts(
    combined: pl.DataFrame, 
    subset: pl.DataFrame, 
    figname=None
) :
    # key: block label
    # value: color
    colors = {
        "Transcriptomics": "#EF476F",
        "Phosphoproteomics": "#F7866B",
        "Proteomics": "#FFD166",
        "Acetylomics": "#06D6A0",
        "Lipidomics": "#11A2B2",
        "Metabolomics": "#0B6584",
        "Filtered": "#444444"
    }

    def _hex_to_rgba(hex: str, alpha: float) -> str : 
        """ add some transparency to rgb hex code and format it how plotly likes it """
        # assume the hex string starts with #
        ir = int(hex[1:3], 16)
        ig = int(hex[3:5], 16)
        ib = int(hex[5:7], 16)
        return f"rgba({ir}, {ig}, {ib}, {alpha})"

    # feature counts 
    counts = (
        combined
        .group_by("Block").len()
        .rename({"len": "pre"})
        .join(
            (
                subset
                .group_by("Block").len()
                .rename({"len": "post"})
            ), 
            how="left", 
            on="Block"
        )
        .fill_null(0)
        .sort("pre", descending=True)
    )

    # node: 
    #   key: {
    #       "index": index, 
    #       "color": color, 
    #       "label": label
    #       "count": count 
    #       "filtered": label <- for source nodes only
    # }
    node_data = {}

    # construct the node data from feature counts pre- and post-subsetting
    i = 0
    filtered = 0
    for block, pre, post in counts.iter_rows() :
        # add the node for pre-subsetting feature count
        node_data[f"{block}_0"] = {
            "index": i,
            "color": colors[block],
            "label": f"{block} ({pre})",
            "count": pre,
            "filtered": pre - post
        }
        filtered += pre - post
        i += 1
        # add the node for post-subsetting feature count
        node_data[f"{block}_1"] = {
            "index": i,
            "color": colors[block],
            "label": f"({post}) {block}",
            "count": post
        }
        i += 1

    # add a catch-all node for features that were filtered out
    node_data["Filtered"] = {
        "index": i,
        "color": colors["Filtered"],
        "label": f"({filtered}) Filtered",
        "count": filtered
    }

    # construct the link data from the node data
    link_data = {"source": [], "target": [], "value": [], "color": []}
    for block, pre, post in counts.iter_rows():
        # if features for a given block remained after subsetting, add a link
        if post > 0:
            link_data["source"].append(node_data[f"{block}_0"]["index"])
            link_data["target"].append(node_data[f"{block}_1"]["index"])
            link_data["value"].append(node_data[f"{block}_1"]["count"])
            link_data["color"].append(_hex_to_rgba(node_data[f"{block}_0"]["color"], 0.25))
        # always add the link from the block to filtered node
        link_data["source"].append(node_data[f"{block}_0"]["index"])
        link_data["target"].append(node_data["Filtered"]["index"])
        link_data["value"].append(node_data[f"{block}_0"]["filtered"])
        link_data["color"].append(_hex_to_rgba(node_data["Filtered"]["color"], 0.25))

        fig = pgo.Figure(
        data=[
            pgo.Sankey(
                arrangement="freeform",
                node=dict(
                    pad=15,
                    thickness=20,
                    line=dict(color="black", width=0.75),
                    label=[v["label"] for v in sorted(node_data.values(), key=lambda x: x["index"])],
                    color=[v["color"] for v in sorted(node_data.values(), key=lambda x: x["index"])],
                ),
                link=link_data
            )
        ]
    )
    if figname is not None:
        fig.write_image(figname, width=600, height=350, scale=4)
    fig.show()

### NPM1 vs. WT
#### Black patients

In [20]:
# select the sample columns
# WT
wt_cols = meta.filter(
    (pl.col("Race") == "Black")
    & ~pl.col("NPM1")
    & ~pl.col("FLT3_ITD")
    & ~pl.col("IDH1+IDH2")
    & ~pl.col("NRAS")
)["Sample"].to_list()
print(f"{wt_cols=}")
# NPM1
mut_cols = meta.filter(
    (pl.col("Race") == "Black")
    & pl.col("NPM1")
    & ~pl.col("FLT3_ITD")
    & ~pl.col("IDH1+IDH2")
    & ~pl.col("NRAS")
)["Sample"].to_list()
print(f"{mut_cols=}")

# save the sample columns
with open(os.path.join(CACHE_DIR, "B_NPM1-WT_cols.pkl"), "wb") as pf:
    pickle.dump((wt_cols, mut_cols), pf)

# subset the omics data
subset = select_subset(combined, wt_cols, mut_cols)

# save the subset to file
subset.write_ipc(os.path.join(CACHE_DIR, "B_NPM1-WT.arrow"))

# display
display(subset)

# report on the number of features in each block
subset.group_by("Block").len().sort("len", descending=True)

# plot feature counts before/after subsetting
plot_filtered_feature_counts(
    combined,
    subset,
    figname=os.path.join(FIGURE_DIR, "block-feature-counts_B_NPM1-WT.png")
)

wt_cols=['14-00240', '16-00538', '16-01109', '17-01060', 'C-05-0664', 'C-04-1820', 'C-07-3682', 'C-10-1211', 'C-10-3429', 'C-08-2091', 'C-13-0276', 'C-08-3337', 'C-10-3924', 'C-09-3512', 'C-09-5462', 'C-08-3493', 'C-04-3407', 'C-09-1074', 'C-09-4769', 'C-10-3906', 'C-07-2070', 'C-05-2179', 'C-05-2448', 'C-07-2900', '94-C-279', 'C-09-1608', 'C-02-0218', 'C-96-117', 'C-95-054', 'C-12-4258', 'C-10-0773', 'C-00-0552', 'C-11-5466', 'C-10-0535', 'C-98-0454', 'C-99-0901', 'C-03-2056', 'C-10-0302', 'C-01-1599', 'C-08-2081', 'C-09-0923', '93-C-201', 'C-01-1998', 'C-99-2136', 'C-05-1782', 'PS88-0050', 'C-01-2163', 'C-02-0350', 'C-01-0171', 'C-99-1077', 'C-05-0004', 'C-05-4372', 'PS89-0158', '94-C-077', 'C-98-0033', 'C-98-0665', 'C-02-1356', '16-01109-Bridge']
mut_cols=['16-00611', '16-01267', 'C-00-1828', 'C-08-0176', 'C-97-0509', 'C-09-1336', 'C-05-3084']


Block,Feature,14-00240,16-00538,16-01109,17-01060,C-05-0664,C-04-1820,C-07-3682,C-10-1211,C-10-3429,C-08-2091,C-13-0276,C-08-3337,C-10-3924,C-09-3512,C-09-5462,C-08-3493,C-04-3407,C-09-1074,C-09-4769,C-10-3906,C-07-2070,C-05-2179,C-05-2448,C-07-2900,94-C-279,C-09-1608,C-02-0218,C-96-117,C-95-054,C-12-4258,C-10-0773,C-00-0552,C-11-5466,C-10-0535,C-98-0454,C-99-0901,C-03-2056,C-10-0302,C-01-1599,C-08-2081,C-09-0923,93-C-201,C-01-1998,C-99-2136,C-05-1782,PS88-0050,C-01-2163,C-02-0350,C-01-0171,C-99-1077,C-05-0004,C-05-4372,PS89-0158,94-C-077,C-98-0033,C-98-0665,C-02-1356,16-01109-Bridge,16-00611,16-01267,C-00-1828,C-08-0176,C-97-0509,C-09-1336,C-05-3084
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Lipidomics""","""neg_LPI 20:4_[M-H]-__A""",0.037803,0.015719,-0.442416,0.015719,-1.644578,-1.535797,-0.849171,-0.129378,-1.895742,-1.687875,-0.618877,-0.588307,-0.995509,-0.514681,-1.507139,-1.917115,-0.085471,-1.144968,0.516209,0.665558,1.431515,0.445511,0.026599,0.905971,-0.764871,0.931503,0.454802,0.645154,-0.929113,-1.10907,3.324811,0.40462,-0.134961,0.221614,0.684336,0.402136,1.007987,0.886675,0.088677,-0.225346,-0.612008,0.122194,0.905151,0.120856,1.037126,-0.112854,1.535687,0.004839,-0.975211,-0.018752,1.312779,1.965016,1.19055,0.578727,1.708673,-0.700252,-1.093332,0.015719,0.015719,0.015719,-1.329567,-0.066051,-1.137861,0.962647,0.161951
"""Lipidomics""","""neg_LPG 22:6_[M-H]-__A""",1.106777,0.157645,0.756149,0.157645,-1.413508,-0.848931,-1.625053,-0.248627,-0.811672,0.472171,1.287657,0.332615,-0.474067,-1.115722,-1.269709,0.445505,-1.371848,-0.255936,1.254133,1.445658,0.602197,0.740431,1.410423,1.615876,0.201448,1.884283,0.954535,-0.713042,-0.969813,0.001223,0.589237,-1.199894,0.782087,-1.001471,1.158672,0.712394,0.309251,0.713028,1.018365,0.16153,0.138358,-3.017299,-0.762418,0.565968,0.113792,-0.006793,-0.479716,-0.619401,-0.647393,0.13419,0.245769,1.728212,0.40435,-1.509973,0.542413,0.834905,-1.196517,0.157645,0.157645,0.157645,-2.515828,0.335503,-0.420464,0.15376,-1.445993
"""Lipidomics""","""neg_LPG 22:5_[M-H]-__A""",-1.003826,0.088184,0.062626,0.088184,-2.431007,-2.154363,-0.826158,0.037789,-0.649627,1.353172,-0.543272,1.275742,-0.764986,-0.756511,0.355321,-0.081711,0.113742,-0.481529,0.340655,0.821486,0.476488,-0.418315,0.601415,1.224892,1.359073,0.510262,0.91419,0.376257,-0.882954,-0.76064,0.900148,-0.110494,0.230001,-0.129398,-0.434228,1.337921,0.614663,1.992648,1.082391,-1.448066,0.718863,-1.154883,0.568836,1.286483,0.048934,-0.514753,0.734121,-1.164928,-1.374675,0.425784,0.249942,1.997585,0.500589,-1.049373,-0.934797,-0.833349,-0.795429,0.088184,0.088184,0.088184,-3.285683,0.676836,0.408397,1.333626,-0.386845
"""Lipidomics""","""neg_LPS 18:1_[M-H]-__A""",0.18351,-0.190818,-0.81828,-0.190818,0.214491,0.474959,-0.718216,1.168354,-0.876946,-0.150536,-1.241234,-0.147042,-2.295903,-0.673824,-0.906904,-1.182887,-0.81274,-1.4339,-0.284411,-0.369393,-0.976312,-0.988105,-1.27147,-0.389334,-0.159276,-0.846201,1.239777,1.384173,-0.424502,-0.802963,1.078401,1.707475,-0.712376,-0.570001,1.146581,1.119198,1.292917,-1.24885,1.430179,-1.16667,-1.577271,0.692627,1.584319,1.643478,-0.340326,-0.121264,2.648348,0.974429,0.379038,0.707395,0.686933,0.297536,0.699017,1.253703,2.321367,-0.222361,0.208935,-0.190818,-0.190818,-0.190818,-0.280096,-0.29137,-0.403257,-0.15379,-0.72504
"""Lipidomics""","""neg_LPI 22:4_[M-H]-__A""",-0.538562,-0.003494,-0.175277,-0.003494,-0.003494,-0.066684,-0.003494,-0.930248,-0.003494,-0.003494,-0.031376,-1.861094,-0.691787,-0.570269,-0.187944,-0.85445,-1.82588,-1.863123,1.226465,0.276049,0.687327,-0.439093,-1.946139,1.251509,-1.300419,-0.343088,0.72661,0.595006,-0.865194,-0.888499,2.270453,1.023759,0.407426,1.105851,0.47141,0.783551,1.536039,0.238351,0.720828,-0.875094,-0.462128,-2.462993,0

In [21]:
(
    combined
    .group_by("Block").len()
    .rename({"len": "pre"})
    .join(
        (
            subset
            .group_by("Block").len()
            .rename({"len": "post"})
        ), 
        how="left", 
        on="Block"
    )
    .fill_null(0)
    .sort("pre", descending=True)
)

Block,pre,post
str,u32,u32
"""Transcriptomics""",14372,0
"""Phosphoproteomics""",8362,4539
"""Proteomics""",8059,7441
"""Acetylomics""",3285,0
"""Lipidomics""",343,325
"""Metabolomics""",49,49


#### White patients 

In [22]:
# select the sample columns
# WT
wt_cols = meta.filter(
    (pl.col("Race") == "White")
    & ~pl.col("NPM1")
    & ~pl.col("FLT3_ITD")
    & ~pl.col("IDH1+IDH2")
    & ~pl.col("NRAS")
)["Sample"].to_list()
print(f"{wt_cols=}")
# NPM1
mut_cols = meta.filter(
    (pl.col("Race") == "White")
    & pl.col("NPM1")
    & ~pl.col("FLT3_ITD")
    & ~pl.col("IDH1+IDH2")
    & ~pl.col("NRAS")
)["Sample"].to_list()
print(f"{mut_cols=}")

# save the sample columns
with open(os.path.join(CACHE_DIR, "W_NPM1-WT_cols.pkl"), "wb") as pf:
    pickle.dump((wt_cols, mut_cols), pf)

# subset the omics data
subset = select_subset(combined, wt_cols, mut_cols)

# save the subset to file
subset.write_ipc(os.path.join(CACHE_DIR, "W_NPM1-WT.arrow"))

# display
display(subset)

# report on the number of features in each block
subset.group_by("Block").len().sort("len", descending=True)

# plot feature counts before/after subsetting
plot_filtered_feature_counts(
    combined,
    subset,
    figname=os.path.join(FIGURE_DIR, "block-feature-counts_W_NPM1-WT.png")
)

wt_cols=['13-00033', '13-00034', '13-00077', '13-00149', '13-00160', '13-00186', '13-00226', '13-00468', '13-00558', '13-00602', '14-00528', '14-00423', '14-00434', '14-00514', '14-00608', '14-00613', '14-00711', '15-00043', '15-00888', '15-00137', '15-00287', '15-00563', '15-00742', '15-00777', '15-00837', '15-00870', '16-00001', '16-00094', '16-00306', '16-00351', '16-00354', '16-00479', '16-00481', '16-00494', '16-00519', '16-00627', '16-00882', '16-00818', '18-00149', '16-01004', '16-01102', '16-01201', '16-01270', '17-00322', '17-00325', '17-00328', '18-00251', '17-00500', '17-00678', '17-00676', '17-00763', '17-00878', '17-00881', '17-00896', '18-00103', '18-00105', '18-00190', '18-00203', '18-00218', '18-00290', '18-00414', '19-00084', '14-00126', '17-00649', '14-00528-Bridge', '16-00494-Bridge', '16-00627-Bridge', '17-00881-Bridge']
mut_cols=['12-00127', '13-00123', '13-00157', '13-00195', '13-00331', '15-00482', '15-00051', '15-00276', '15-00302', '15-00653', '15-00855', '16-0

Block,Feature,13-00033,13-00034,13-00077,13-00149,13-00160,13-00186,13-00226,13-00468,13-00558,13-00602,14-00528,14-00423,14-00434,14-00514,14-00608,14-00613,14-00711,15-00043,15-00888,15-00137,15-00287,15-00563,15-00742,15-00777,15-00837,15-00870,16-00001,16-00094,16-00306,16-00351,16-00354,16-00479,16-00481,16-00494,16-00519,…,18-00218,18-00290,18-00414,19-00084,14-00126,17-00649,14-00528-Bridge,16-00494-Bridge,16-00627-Bridge,17-00881-Bridge,12-00127,13-00123,13-00157,13-00195,13-00331,15-00482,15-00051,15-00276,15-00302,15-00653,15-00855,16-00056,16-00886,16-00504,16-01061,16-01100,16-01151,16-01227,17-00021,17-00361,17-00444,17-00689,17-00770,17-00834,17-01054,19-00092,16-01100-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Proteomics""","""A1BG""",-1.120712,0.559693,0.509441,-1.092024,2.115176,0.505511,-0.35066,-2.292624,0.755917,0.99725,0.887855,-0.536614,0.202024,-2.195447,-0.975754,0.643801,1.395539,-0.364754,-1.036813,-0.066231,-1.038048,0.11264,-0.752465,-0.080969,-0.080969,1.329033,0.17818,-0.325316,-0.337432,0.601645,-0.080969,0.822556,-0.832914,0.369056,-0.080969,…,0.299175,-0.087081,-0.080969,2.882484,0.742085,1.121497,-1.986445,-0.488179,-0.769835,-0.368235,-0.080969,-0.080969,0.629619,1.418677,-1.031601,-0.292654,-1.671648,-0.348888,-0.080969,-0.16544,-0.916447,-0.080969,1.659495,-0.670049,-0.248601,0.326066,-0.677482,0.58812,0.186894,2.165707,1.577976,-1.197035,-0.080969,2.123743,-1.570688,-0.158679,-0.729561
"""Proteomics""","""A2M""",-0.645049,1.652101,0.656217,-1.223615,2.07074,0.046133,-1.360862,1.261334,-0.981207,1.989444,1.38015,1.126968,-0.195347,-0.404027,-1.129677,0.870149,1.177509,0.702726,-1.030546,0.027664,0.78199,-0.095451,-0.281378,0.795514,-0.812216,-0.042882,-0.56069,0.077096,-1.198138,3.00129,0.826018,-0.481681,-0.871139,0.443083,-1.59836,…,-0.627944,-1.134956,-1.086976,2.686582,-0.359811,0.421092,-0.80656,-0.551992,-0.590998,-0.040188,-1.296593,1.649479,0.563176,0.841247,-1.111923,-0.492714,-2.034394,-0.369432,-0.638296,-0.351672,1.162318,-1.475426,0.379921,1.275077,-0.140934,1.126538,-0.696944,0.422628,-0.712251,0.289353,0.609577,-1.479869,1.836071,0.912377,-0.551094,-0.814283,-0.361819
"""Proteomics""","""AAAS""",0.030761,1.399064,0.609426,-0.928379,0.022328,-0.034376,-2.402319,-2.01744,-0.151756,1.633452,1.111318,0.128096,0.451588,-0.445137,-1.585541,0.249108,0.703759,-0.767651,-0.951632,0.372298,-0.203209,0.689281,0.78313,1.367979,-1.906336,0.821236,0.847545,0.250625,-1.029745,-1.469132,-1.388439,-0.422609,-0.437779,-0.077003,-0.563093,…,0.447461,-0.073484,-0.663477,-2.157879,-0.774188,1.068962,-0.604054,1.555145,-2.226617,-0.308976,-0.462863,1.71303,1.510332,0.870591,0.031587,-0.76457,-1.396271,0.273021,-1.288161,-0.16878,-0.667412,-0.94183,1.262147,-2.578698,0.792216,0.615189,1.132389,0.634947,-0.153854,0.237538,0.748446,-1.155784,0.940188,0.686234,-1.342254,-0.947444,0.503089
"""Proteomics""","""AACS""",0.07813,1.898765,0.930045,-1.195709,-0.137816,-0.481944,-2.371388,-1.402506,-0.01819,1.249069,0.881115,-0.106658,0.724629,-0.002164,-0.790637,-0.337542,0.325305,-0.404858,-0.670801,-0.469277,-0.528342,0.316998,1.16019,-0.319149,-1.127131,-0.492388,-0.252296,-0.645888,-0.958213,-0.906817,-2.00887,0.549092,-0.717205,1.215256,-1.148569,…,-0.227115,0.253181,-1.069696,-2.069872,-0.591218,-0.019467,0.644242,1.989462,-0.044668,0.333034,-0.841923,1.992072,0.948301,1.266173,-0.681241,-0.249042,-0.379809,1.604692,-0.735021,-0.040774,0.370725,-1.708815,1.679816,-1.918284,0.689695,1.455438,1.281812,-0.846586,-0.15247,1.074701,1.083111,-0.727126,0.736447,0.944014,0.197232,-0.234734,2.155406
"""Proteomics""","""AAGAB""",-0.007986,1.871435,1.174142,-0.736959,0.470591,0.505011,-2.042391,-1.716918,-0.307311,1.478371,0.697132,-0.319659,0.737242,-0

### NRAS vs. WT
#### Black patients

In [23]:
# select the sample columns
# WT
wt_cols = meta.filter(
    (pl.col("Race") == "Black")
    & ~pl.col("NPM1")
    & ~pl.col("FLT3_ITD")
    & ~pl.col("IDH1+IDH2")
    & ~pl.col("NRAS")
)["Sample"].to_list()
print(f"{wt_cols=}")
# NRAS
mut_cols = meta.filter(
    (pl.col("Race") == "Black")
    & ~pl.col("NPM1")
    & ~pl.col("FLT3_ITD")
    & ~pl.col("IDH1+IDH2")
    & pl.col("NRAS")
)["Sample"].to_list()
print(f"{mut_cols=}")

# save the sample columns
with open(os.path.join(CACHE_DIR, "B_NRAS-WT_cols.pkl"), "wb") as pf:
    pickle.dump((wt_cols, mut_cols), pf)

# subset the omics data
subset = select_subset(combined, wt_cols, mut_cols)

# save the subset to file
subset.write_ipc(os.path.join(CACHE_DIR, "B_NRAS-WT.arrow"))

# display
display(subset)

# report on the number of features in each block
subset.group_by("Block").len().sort("len", descending=True)

# plot feature counts before/after subsetting
plot_filtered_feature_counts(
    combined,
    subset,
    figname=os.path.join(FIGURE_DIR, "block-feature-counts_B_NRAS-WT.png")
)

wt_cols=['14-00240', '16-00538', '16-01109', '17-01060', 'C-05-0664', 'C-04-1820', 'C-07-3682', 'C-10-1211', 'C-10-3429', 'C-08-2091', 'C-13-0276', 'C-08-3337', 'C-10-3924', 'C-09-3512', 'C-09-5462', 'C-08-3493', 'C-04-3407', 'C-09-1074', 'C-09-4769', 'C-10-3906', 'C-07-2070', 'C-05-2179', 'C-05-2448', 'C-07-2900', '94-C-279', 'C-09-1608', 'C-02-0218', 'C-96-117', 'C-95-054', 'C-12-4258', 'C-10-0773', 'C-00-0552', 'C-11-5466', 'C-10-0535', 'C-98-0454', 'C-99-0901', 'C-03-2056', 'C-10-0302', 'C-01-1599', 'C-08-2081', 'C-09-0923', '93-C-201', 'C-01-1998', 'C-99-2136', 'C-05-1782', 'PS88-0050', 'C-01-2163', 'C-02-0350', 'C-01-0171', 'C-99-1077', 'C-05-0004', 'C-05-4372', 'PS89-0158', '94-C-077', 'C-98-0033', 'C-98-0665', 'C-02-1356', '16-01109-Bridge']
mut_cols=['C-98-0846', 'C-09-1906', 'C-05-1287', 'C-04-1098', 'C-09-5381', 'C-05-0569', 'C-04-1647', 'C-99-0740', 'C-12-2943', 'C-95-068', 'C-01-1665', 'C-05-3159', 'C-98-0031']


Block,Feature,14-00240,16-00538,16-01109,17-01060,C-05-0664,C-04-1820,C-07-3682,C-10-1211,C-10-3429,C-08-2091,C-13-0276,C-08-3337,C-10-3924,C-09-3512,C-09-5462,C-08-3493,C-04-3407,C-09-1074,C-09-4769,C-10-3906,C-07-2070,C-05-2179,C-05-2448,C-07-2900,94-C-279,C-09-1608,C-02-0218,C-96-117,C-95-054,C-12-4258,C-10-0773,C-00-0552,C-11-5466,C-10-0535,C-98-0454,C-99-0901,C-03-2056,C-10-0302,C-01-1599,C-08-2081,C-09-0923,93-C-201,C-01-1998,C-99-2136,C-05-1782,PS88-0050,C-01-2163,C-02-0350,C-01-0171,C-99-1077,C-05-0004,C-05-4372,PS89-0158,94-C-077,C-98-0033,C-98-0665,C-02-1356,16-01109-Bridge,C-98-0846,C-09-1906,C-05-1287,C-04-1098,C-09-5381,C-05-0569,C-04-1647,C-99-0740,C-12-2943,C-95-068,C-01-1665,C-05-3159,C-98-0031
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Acetylomics""","""ABCE1-K431k""",0.001422,0.001422,-1.166789,0.001422,0.001422,0.001422,0.001422,0.001422,0.001422,0.001422,-0.980066,-2.042584,-1.501437,-0.119523,-1.740564,-0.037087,-2.950525,0.721613,0.433269,-0.423469,-0.308268,0.512584,0.58693,-0.610934,0.033375,-0.790028,0.801723,-0.964853,-0.97536,-0.292663,0.370716,0.472016,-0.745587,-0.02767,0.718426,2.211175,2.256342,1.012737,-0.160991,-2.275488,-1.030277,0.508378,-0.335413,1.080801,0.998154,0.689513,-1.277879,0.889109,0.001422,0.819578,0.09433,0.578357,1.068374,-0.239302,-1.419909,2.652778,0.585864,0.001422,0.269435,-0.023325,-0.081494,0.552655,-1.480116,0.165458,0.846122,0.660477,-0.485424,-0.904046,2.001628,-0.072301,0.855812
"""Acetylomics""","""ACAA2-K137k""",-0.105592,-0.105592,-0.222003,-0.105592,-0.105592,-0.105592,-0.105592,-0.105592,-0.105592,-0.105592,0.087653,0.814877,1.110306,-0.305182,0.300468,-0.399479,1.482062,-0.61605,1.842759,-0.38774,1.370462,0.756178,1.261365,1.420956,-0.404725,-0.105592,0.814327,-3.943706,1.056956,-0.411506,2.594053,1.193382,2.290586,-1.419048,-0.073074,-0.717161,0.889877,-0.514555,-0.527671,1.625991,-0.753024,-0.032555,-1.567162,-0.751535,0.326644,-0.062016,-0.489348,-0.748716,-0.632677,-0.886818,-0.917087,0.445746,-1.790275,-0.160219,-0.150027,-1.166248,1.014331,-0.105592,1.032799,-0.545849,0.731329,0.025474,-0.322703,-0.092919,0.189691,-0.582366,-0.299096,0.235691,0.394843,-1.908605,-0.346154
"""Acetylomics""","""ACAA2-K209k""",0.102067,0.102067,-0.811759,0.102067,0.102067,0.102067,0.102067,0.102067,0.102067,0.102067,-0.980511,0.573117,-0.821314,-0.007486,0.102067,1.008038,-1.192998,0.017314,0.316147,-2.467833,-0.57151,1.036609,0.206152,2.185928,1.101731,0.512332,0.256866,-1.232475,-1.331842,-0.990088,2.483699,1.794139,0.798921,-1.3801,-0.428255,-0.436052,0.856124,0.783122,1.080063,0.416642,-1.056804,0.449306,-2.523506,-0.12433,-0.630713,0.733986,-0.4535,0.80653,0.102067,-0.833652,-0.375918,2.005933,-0.266946,0.72677,0.69039,0.569653,0.705889,0.102067,-0.306912,0.102067,0.448472,0.23799,-2.31612,1.357824,-0.442203,-0.699109,0.94253,0.107032,-1.253671,-2.356755,-0.243761
"""Acetylomics""","""ACAA2-K234k""",-0.08221,-0.08221,0.950628,-0.08221,-0.08221,-0.08221,-0.08221,-0.08221,-0.08221,-0.08221,1.530123,1.279716,1.659776,0.416097,0.8437,-0.081871,1.649894,-2.611844,1.764079,-0.572147,0.903818,-0.15435,0.162393,0.784704,-1.460268,-0.691743,-0.44019,-0.08221,-0.174716,-1.093294,1.854218,1.006636,2.055081,-1.698016,-0.737434,-1.530963,0.688507,1.112864,-0.63461,1.039078,-0.870652,-0.339957,-1.948692,-0.816976,1.128135,-0.212858,-1.393212,-0.296109,0.571988,-0.111116,-1.215611,0.07281,-0.943191,-0.64427,0.436402,-0.936731,2.011001,-0.08221,0.753603,1.564511,-0.082549,-0.203294,-1.460056,0.420112,0.047111,-0.83091,0.020928,-0.579465,0.243256,-1.657397,0.357634
"""Acetylomics""","""ACAA2-K269k""",0.105122,0.105122,0.511866,0.105122,0.105122,0.105122,0.105122,0.105122,0.105122,0.105122,-0.636443,0.991685,0.951769,0.241325,0.249329,-0.47295

#### White patients

In [24]:
# select the sample columns
# WT
wt_cols = meta.filter(
    (pl.col("Race") == "White")
    & ~pl.col("NPM1")
    & ~pl.col("FLT3_ITD")
    & ~pl.col("IDH1+IDH2")
    & ~pl.col("NRAS")
)["Sample"].to_list()
print(f"{wt_cols=}")
# NRAS
mut_cols = meta.filter(
    (pl.col("Race") == "White")
    & ~pl.col("NPM1")
    & ~pl.col("FLT3_ITD")
    & ~pl.col("IDH1+IDH2")
    & pl.col("NRAS")
)["Sample"].to_list()
print(f"{mut_cols=}")

# save the sample columns
with open(os.path.join(CACHE_DIR, "W_NRAS-WT_cols.pkl"), "wb") as pf:
    pickle.dump((wt_cols, mut_cols), pf)

# subset the omics data
subset = select_subset(combined, wt_cols, mut_cols)

# save the subset to file
subset.write_ipc(os.path.join(CACHE_DIR, "W_NRAS-WT.arrow"))

# display
display(subset)

# report on the number of features in each block
subset.group_by("Block").len().sort("len", descending=True)

# plot feature counts before/after subsetting
plot_filtered_feature_counts(
    combined,
    subset,
    figname=os.path.join(FIGURE_DIR, "block-feature-counts_W_NRAS-WT.png")
)

wt_cols=['13-00033', '13-00034', '13-00077', '13-00149', '13-00160', '13-00186', '13-00226', '13-00468', '13-00558', '13-00602', '14-00528', '14-00423', '14-00434', '14-00514', '14-00608', '14-00613', '14-00711', '15-00043', '15-00888', '15-00137', '15-00287', '15-00563', '15-00742', '15-00777', '15-00837', '15-00870', '16-00001', '16-00094', '16-00306', '16-00351', '16-00354', '16-00479', '16-00481', '16-00494', '16-00519', '16-00627', '16-00882', '16-00818', '18-00149', '16-01004', '16-01102', '16-01201', '16-01270', '17-00322', '17-00325', '17-00328', '18-00251', '17-00500', '17-00678', '17-00676', '17-00763', '17-00878', '17-00881', '17-00896', '18-00103', '18-00105', '18-00190', '18-00203', '18-00218', '18-00290', '18-00414', '19-00084', '14-00126', '17-00649', '14-00528-Bridge', '16-00494-Bridge', '16-00627-Bridge', '17-00881-Bridge']
mut_cols=['14-00127', '14-00193', '14-00712', '15-00377', '15-00578', '15-00813', '15-00976', '16-00751', '16-01191', '16-01220', '17-00072', '17-0

Block,Feature,13-00033,13-00034,13-00077,13-00149,13-00160,13-00186,13-00226,13-00468,13-00558,13-00602,14-00528,14-00423,14-00434,14-00514,14-00608,14-00613,14-00711,15-00043,15-00888,15-00137,15-00287,15-00563,15-00742,15-00777,15-00837,15-00870,16-00001,16-00094,16-00306,16-00351,16-00354,16-00479,16-00481,16-00494,16-00519,…,17-00328,18-00251,17-00500,17-00678,17-00676,17-00763,17-00878,17-00881,17-00896,18-00103,18-00105,18-00190,18-00203,18-00218,18-00290,18-00414,19-00084,14-00126,17-00649,14-00528-Bridge,16-00494-Bridge,16-00627-Bridge,17-00881-Bridge,14-00127,14-00193,14-00712,15-00377,15-00578,15-00813,15-00976,16-00751,16-01191,16-01220,17-00072,17-00901,18-00408,16-01191-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Proteomics""","""A1BG""",-1.121872,0.595768,0.544402,-1.092548,2.185717,0.540385,-0.334758,-2.319751,0.79634,1.04302,0.931201,-0.524831,0.230173,-2.220421,-0.973703,0.681739,1.450134,-0.349163,-1.036114,-0.044026,-1.037377,0.138809,-0.745466,-0.048434,-0.048434,1.382154,0.205801,-0.308852,-0.321236,0.638649,-0.048434,0.864456,-0.827697,0.400906,-0.048434,…,0.705287,1.596714,0.11649,-0.048434,-0.131598,-0.048434,1.340252,0.475523,-2.472768,1.044669,0.91106,1.164539,-1.090254,0.329477,-0.065338,-0.048434,2.970027,0.782201,1.17002,-2.006788,-0.475323,-0.76322,-0.352722,-0.048434,0.313661,-0.048434,-0.26305,-0.409594,-0.730241,-0.346546,0.724103,1.577289,-2.604304,-0.112906,-0.048434,-0.048434,0.956138
"""Proteomics""","""A2M""",-0.65365,1.698652,0.678858,-1.246107,2.127343,0.054127,-1.386649,1.298504,-0.997879,2.044095,1.420173,1.160912,-0.193151,-0.406841,-1.149913,0.897927,1.212666,0.726484,-1.048402,0.035214,0.807651,-0.090857,-0.281248,0.8215,-0.824831,-0.037025,-0.567265,0.085833,-1.220018,3.080234,0.852736,-0.48636,-0.885168,0.460607,-1.629849,…,-0.270572,-0.367764,0.250711,-0.995062,-0.129135,0.133184,0.071536,1.192127,-0.137078,1.930023,0.636489,0.50796,0.109818,-0.636135,-1.15532,-1.106187,2.75797,-0.361563,0.438088,-0.819039,-0.558359,-0.598301,-0.034267,-0.488027,0.595656,0.780405,-0.185509,0.159927,0.482323,-1.935348,-0.146521,0.594961,-1.974985,-0.591918,1.040626,0.094029,-0.388813
"""Proteomics""","""AAAS""",0.064053,1.430991,0.642141,-0.89413,0.055629,-0.001019,-2.3666,-1.982105,-0.118282,1.665145,1.143533,0.161291,0.484461,-0.41137,-1.550636,0.282182,0.73638,-0.733562,-0.917359,0.405249,-0.169683,0.721916,0.815672,1.399937,-1.871111,0.853739,0.880022,0.283698,-0.995395,-1.434343,-1.353731,-0.388865,-0.40402,-0.043603,-0.529209,…,0.140731,-0.084765,1.066742,-0.585976,1.169106,1.676812,1.038752,0.251748,-0.477276,1.249834,0.087302,1.418589,-0.556626,0.480338,-0.040088,-0.629492,-2.122403,-0.740093,1.101218,-0.570128,1.586916,-2.191072,-0.275345,0.485239,0.320922,0.59039,0.829727,1.338993,0.233603,-2.018841,-0.291711,-0.060883,-1.59401,0.725258,0.518186,-1.455829,-1.804064
"""Proteomics""","""AACS""",0.170445,2.119524,1.082462,-1.193263,-0.060736,-0.429141,-2.451884,-1.414648,0.06733,1.423993,1.03008,-0.027379,0.862553,0.084486,-0.759613,-0.274553,0.435058,-0.346617,-0.631322,-0.415581,-0.478813,0.426165,1.328843,-0.254861,-1.119846,-0.440322,-0.183292,-0.604651,-0.939011,-0.883989,-2.06379,0.674633,-0.681,1.387794,-1.142797,…,0.60618,-0.671797,2.167131,-0.593033,-1.256561,1.674249,0.549525,0.109126,-0.864576,0.463025,-0.925079,2.204697,-0.72038,-0.156335,0.357846,-1.058359,-2.129097,-0.546125,0.065962,0.776496,2.21662,0.038984,0.443332,1.348273,0.184427,1.063325,-0.459502,0.70708,0.131157,-0.561707,-1.085141,0.705875,-1.512223,0.418716,2.244499,-0.256737,0.765562
"""Proteomics""","""AAGAB""",-0.00279,1.917118,1.204803,-0.747465,0.486097,0.521259,-2.081019,-1.748535,-0.308563,1.515586,0.717517,-0.321176,0.758492,-0.641651,-1.588542,-0.1